In [1]:
"""
STEP 2: Train VQ-VAE Tokenizer for D=7 (M750 + UMI gripper)
=============================================================
Shard format (confirmed):
  Each .tar file contains per-frame entries:
    XXXXXX.action.npy       → shape (24, 7) float32  ← we use this
    XXXXXX.action_token.npy → shape (27,)   int16
    XXXXXX.image.jpg        → wrist camera image
    XXXXXX.meta.json        → episode/frame info

Run from RDT2 directory:
    cd /home/rishabh/Downloads/umi-pipeline-training/RDT2
    source /home/rishabh/Downloads/umi-pipeline-training/umi_env/bin/activate
    python step2_train_vqvae.py
"""

import os, sys, glob, tarfile, io, torch
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── CONFIG ────────────────────────────────────────────────────────────────────
SHARDS_DIR  = "/home/rishabh/Downloads/umi-pipeline-training/shards"
OUTPUT_DIR  = "/home/rishabh/Downloads/umi-pipeline-training/outputs/vqvae-m750-7dof"
RDT2_DIR    = "/home/rishabh/Downloads/umi-pipeline-training/RDT2"
ACTION_DIM  = 7       # [x, y, z, rx, ry, rz, gripper]
ACTION_HZ   = 24      # chunk size — already 24 in your data ✅
BATCH_SIZE  = 64
EPOCHS      = 100
LR          = 1e-4
DEVICE      = "cuda:0" if torch.cuda.is_available() else "cpu"

sys.path.insert(0, RDT2_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Device : {DEVICE}")
print(f"Shards : {SHARDS_DIR}")
print(f"Output : {OUTPUT_DIR}")

# ── STEP 1: PATCH multivqvae.py FOR D=7 ──────────────────────────────────────
print("\n[1/4] Patching multivqvae.py for D=7...")

import shutil
VQVAE_FILE = f"{RDT2_DIR}/vqvae/models/multivqvae.py"
if not os.path.exists(VQVAE_FILE + ".bak_original"):
    shutil.copy(VQVAE_FILE, VQVAE_FILE + ".bak_original")
    print(f"  Backed up original → {VQVAE_FILE}.bak_original")

D7_CODE = '''"""
multivqvae.py — PATCHED FOR M750 SINGLE ARM D=7
================================================
D=7 layout: [x, y, z, rx, ry, rz, gripper]
              0  1  2   3   4   5      6
"""
from typing import Union
import torch
import torch.nn as nn
from huggingface_hub import PyTorchModelHubMixin
from vqvae.models.vqvae import VQVAE


def select_act_dim(x, act_type):
    """Extract sub-dims from D=7 tensor."""
    if act_type == "pos":    return x[..., 0:3]   # xyz
    elif act_type == "rot":  return x[..., 3:6]   # rx,ry,rz
    elif act_type == "grip": return x[..., 6:7]   # gripper
    raise ValueError(f"Unknown act_type: {act_type}")


def apply_act_dim(x, y, act_type):
    """Write sub-dims back into D=7 tensor (in-place)."""
    if act_type == "pos":    x[..., 0:3] = y
    elif act_type == "rot":  x[..., 3:6] = y
    elif act_type == "grip": x[..., 6:7] = y
    return x


class MultiVQVAE(nn.Module, PyTorchModelHubMixin):
    def __init__(self, input_dim, embedding_dim, cnn_config,
                 num_embeddings, action_horizon,
                 n_codebooks=None,
                 codebook_restart_interval=64,
                 commitment_cost=0.25,
                 codebook_cost=0.0, local_rank=0):
        super().__init__()
        if n_codebooks is None:
            n_codebooks = {"pos": 6, "rot": 2, "grip": 1}

        self.pos_vqvae = VQVAE(
            input_dim=input_dim["pos"],
            embedding_dim=embedding_dim,
            cnn_config=cnn_config,
            num_embeddings=num_embeddings,
            action_horizon=action_horizon,
            n_codebooks=n_codebooks["pos"],
            codebook_restart_interval=codebook_restart_interval,
            commitment_cost=commitment_cost,
            codebook_cost=codebook_cost,
            local_rank=local_rank)
        self.pos_id_len = n_codebooks["pos"] * 3   # 18

        self.rot_vqvae = VQVAE(
            input_dim=input_dim["rot"],
            embedding_dim=embedding_dim,
            cnn_config=cnn_config,
            num_embeddings=num_embeddings,
            action_horizon=action_horizon,
            n_codebooks=n_codebooks["rot"],
            codebook_restart_interval=codebook_restart_interval,
            commitment_cost=commitment_cost,
            codebook_cost=codebook_cost,
            local_rank=local_rank)
        self.rot_id_len = n_codebooks["rot"] * 3   # 6

        self.grip_vqvae = VQVAE(
            input_dim=input_dim["grip"],
            embedding_dim=embedding_dim,
            cnn_config=cnn_config,
            num_embeddings=num_embeddings,
            action_horizon=action_horizon,
            n_codebooks=n_codebooks["grip"],
            codebook_restart_interval=codebook_restart_interval,
            commitment_cost=commitment_cost,
            codebook_cost=codebook_cost,
            local_rank=local_rank)
        self.grip_id_len = n_codebooks["grip"] * 3  # 3

        self.action_dim     = input_dim["pos"] + input_dim["rot"] + input_dim["grip"]  # 7
        self.action_horizon = action_horizon
        self.num_embeddings = num_embeddings

    def encode(self, x):
        """x: (B, T, 7) → ids: (B, 27)"""
        if isinstance(x, dict):
            p, r, g = x["pos"], x["rot"], x["grip"]
        else:
            p = select_act_dim(x, "pos")
            r = select_act_dim(x, "rot")
            g = select_act_dim(x, "grip")
        return torch.cat([
            self.pos_vqvae.encode(p),
            self.rot_vqvae.encode(r),
            self.grip_vqvae.encode(g)], dim=-1)

    def decode(self, ids, return_dict=False):
        """ids: (B, 27) → x_recon: (B, T, 7)"""
        p_ids = ids[..., :self.pos_id_len]
        r_ids = ids[..., self.pos_id_len:self.pos_id_len + self.rot_id_len]
        g_ids = ids[..., self.pos_id_len + self.rot_id_len:]
        p = self.pos_vqvae.decode(p_ids)
        r = self.rot_vqvae.decode(r_ids)
        g = self.grip_vqvae.decode(g_ids)
        if return_dict:
            return {"pos": p, "rot": r, "grip": g}
        out = torch.zeros(ids.shape[0], self.action_horizon,
                          self.action_dim, device=ids.device)
        out = apply_act_dim(out, p, "pos")
        out = apply_act_dim(out, r, "rot")
        out = apply_act_dim(out, g, "grip")
        return out  # (B, T, 7)

    def calculate_loss(self, x, x_recon):
        return {
            "pos":  self.pos_vqvae.calculate_loss(
                        select_act_dim(x, "pos"),
                        select_act_dim(x_recon, "pos"), act_type="pos"),
            "rot":  self.rot_vqvae.calculate_loss(
                        select_act_dim(x, "rot"),
                        select_act_dim(x_recon, "rot"), act_type="rot"),
            "grip": self.grip_vqvae.calculate_loss(
                        select_act_dim(x, "grip"),
                        select_act_dim(x_recon, "grip"), act_type="grip"),
        }
'''

with open(VQVAE_FILE, "w") as fh:
    fh.write(D7_CODE)
print("  ✅ multivqvae.py patched for D=7")

# ── STEP 2: DATASET ───────────────────────────────────────────────────────────
print("\n[2/4] Loading M750 shard dataset (from .tar files)...")

class M750TarDataset(Dataset):
    """
    Reads action chunks directly from .tar shards.
    Each tar contains frames like:
        000044.action.npy  → shape (24, 7) float32  ← use directly
        000044.image.jpg   → wrist camera (used in step4)
        000044.meta.json   → episode/frame info
    """
    def __init__(self, shards_dir):
        self.actions = []

        tar_files = sorted(glob.glob(f"{shards_dir}/*.tar"))
        print(f"  Found {len(tar_files)} .tar files")

        skipped = 0
        for tar_path in tar_files:
            try:
                with tarfile.open(tar_path, "r") as tar:
                    for member in tar.getmembers():
                        # Only load action files
                        if not member.name.endswith(".action.npy"):
                            continue
                        try:
                            raw  = tar.extractfile(member).read()
                            arr  = np.load(io.BytesIO(raw))   # (24, 7)
                            arr  = arr.astype(np.float32)

                            # Validate shape
                            if arr.shape != (ACTION_HZ, ACTION_DIM):
                                skipped += 1
                                continue

                            self.actions.append(
                                torch.from_numpy(arr))  # (24, 7)
                        except Exception as e:
                            skipped += 1
            except Exception as e:
                print(f"  Skip tar {tar_path}: {e}")

        print(f"  Loaded : {len(self.actions)} action chunks")
        print(f"  Skipped: {skipped}")

        if len(self.actions) == 0:
            raise RuntimeError(
                f"No valid (24,7) action data found in {shards_dir}!\n"
                f"Check that .tar files contain XXXXXX.action.npy")

        # Print data statistics
        all_acts = torch.stack(self.actions).reshape(-1, ACTION_DIM)
        names = ["x(m)","y(m)","z(m)","rx","ry","rz","grip"]
        print(f"\n  {'dim':<10} {'min':>9} {'max':>9} {'mean':>9} {'std':>9}")
        print(f"  {'-'*50}")
        for i, n in enumerate(names):
            print(f"  {n:<10} "
                  f"{all_acts[:,i].min().item():>9.4f} "
                  f"{all_acts[:,i].max().item():>9.4f} "
                  f"{all_acts[:,i].mean().item():>9.4f} "
                  f"{all_acts[:,i].std().item():>9.4f}")

    def __len__(self):
        return len(self.actions)

    def __getitem__(self, idx):
        return self.actions[idx]  # (24, 7)


dataset = M750TarDataset(SHARDS_DIR)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE,
                     shuffle=True, num_workers=4,
                     pin_memory=True, drop_last=True)
print(f"\n  Batches per epoch: {len(loader)}")

# ── STEP 3: BUILD VQ-VAE ──────────────────────────────────────────────────────
print("\n[3/4] Building VQ-VAE (D=7)...")

from vqvae.models.multivqvae import MultiVQVAE

VQVAE_CONFIG = {
    "input_dim":     {"pos": 3, "rot": 3, "grip": 1},
    "embedding_dim": 64,
    "cnn_config": {
        "channels": [64, 128, 256],
        "kernels":  [4, 4, 4],
        "strides":  [2, 2, 2],
    },
    "num_embeddings":  512,
    "action_horizon":  ACTION_HZ,        # 24
    "n_codebooks":     {"pos": 6, "rot": 2, "grip": 1},
}

vqvae = MultiVQVAE(**VQVAE_CONFIG).to(DEVICE)
n_params = sum(p.numel() for p in vqvae.parameters())
print(f"  action_dim   = {vqvae.action_dim}")     # 7
print(f"  pos_id_len   = {vqvae.pos_id_len}")     # 18
print(f"  rot_id_len   = {vqvae.rot_id_len}")     # 6
print(f"  grip_id_len  = {vqvae.grip_id_len}")    # 3
print(f"  total_id_len = {vqvae.pos_id_len + vqvae.rot_id_len + vqvae.grip_id_len}")  # 27 ✅
print(f"  parameters   = {n_params:,}")

# Quick sanity check
dummy = torch.randn(2, ACTION_HZ, ACTION_DIM).to(DEVICE)
ids   = vqvae.encode(dummy)
recon = vqvae.decode(ids)
assert ids.shape   == (2, 27),              f"Expected (2,27) got {ids.shape}"
assert recon.shape == (2, ACTION_HZ, ACTION_DIM), f"Expected (2,24,7) got {recon.shape}"
print(f"  ✅ Sanity check passed: encode (2,24,7)→(2,27), decode (2,27)→(2,24,7)")

# ── STEP 4: TRAIN ─────────────────────────────────────────────────────────────
print(f"\n[4/4] Training VQ-VAE for {EPOCHS} epochs...")

optimizer = torch.optim.Adam(vqvae.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-5)

best_loss = float("inf")

for epoch in range(EPOCHS):
    vqvae.train()
    totals = {"pos": 0., "rot": 0., "grip": 0., "total": 0.}
    n_batches = 0

    for batch in loader:
        batch = batch.to(DEVICE)             # (B, 24, 7)
        ids   = vqvae.encode(batch)          # (B, 27)
        recon = vqvae.decode(ids)            # (B, 24, 7)
        losses = vqvae.calculate_loss(batch, recon)

        loss = sum(v["loss"] for v in losses.values())

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(vqvae.parameters(), 1.0)
        optimizer.step()

        totals["total"] += loss.item()
        for k in ["pos", "rot", "grip"]:
            totals[k] += losses[k]["loss"].item()
        n_batches += 1

    scheduler.step()
    avg = {k: v / n_batches for k, v in totals.items()}

    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:>3}/{EPOCHS} | "
              f"total={avg['total']:.4f} | "
              f"pos={avg['pos']:.4f} | "
              f"rot={avg['rot']:.4f} | "
              f"grip={avg['grip']:.4f}")

    # Save every 25 epochs
    if (epoch + 1) % 25 == 0:
        ckpt_path = f"{OUTPUT_DIR}/vqvae_epoch{epoch+1}.pt"
        torch.save({
            "epoch":  epoch + 1,
            "model":  vqvae.state_dict(),
            "config": VQVAE_CONFIG,
            "loss":   avg["total"],
        }, ckpt_path)
        print(f"  💾 Saved: {ckpt_path}")

    # Save best
    if avg["total"] < best_loss:
        best_loss = avg["total"]
        torch.save({
            "epoch":  epoch + 1,
            "model":  vqvae.state_dict(),
            "config": VQVAE_CONFIG,
            "loss":   best_loss,
        }, f"{OUTPUT_DIR}/vqvae_best.pt")

# Final save
torch.save({
    "epoch":  EPOCHS,
    "model":  vqvae.state_dict(),
    "config": VQVAE_CONFIG,
    "loss":   avg["total"],
}, f"{OUTPUT_DIR}/vqvae_final.pt")

print(f"\n{'='*55}")
print(f"✅ VQ-VAE training complete!")
print(f"   Best loss : {best_loss:.4f}")
print(f"   Saved to  : {OUTPUT_DIR}/vqvae_final.pt")
print(f"{'='*55}")
print(f"\nNext: Run step3_build_normalizer.py")

Device : cuda:0
Shards : /home/rishabh/Downloads/umi-pipeline-training/shards
Output : /home/rishabh/Downloads/umi-pipeline-training/outputs/vqvae-m750-7dof

[1/4] Patching multivqvae.py for D=7...
  ✅ multivqvae.py patched for D=7

[2/4] Loading M750 shard dataset (from .tar files)...
  Found 740 .tar files
  Loaded : 73986 action chunks
  Skipped: 0

  dim              min       max      mean       std
  --------------------------------------------------
  x(m)         -0.7661    0.3858    0.0089    0.1607
  y(m)         -0.8740    0.2910   -0.0979    0.1369
  z(m)         -0.0393    0.3783    0.1209    0.1024
  rx           -3.1406    3.1409   -2.1489    1.4194
  ry           -3.1398    3.1406   -0.1721    0.8444
  rz           -0.8489    1.1756    0.0470    0.1910
  grip          0.0065    0.0438    0.0260    0.0120

  Batches per epoch: 1156

[3/4] Building VQ-VAE (D=7)...


/home/rishabh/Downloads/umi-pipeline-training/umi_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyError: 'output_size'